In [ ]:
!pip uninstall -y diffusers transformers huggingface-hub tokenizers

!pip install -q \
    "diffusers==0.36.0" \
    "transformers==4.57.3" \
    "huggingface-hub==0.36.2" \
    "tokenizers==0.22.2" \
    accelerate safetensors sentencepiece protobuf

In [ ]:
import torch
import diffusers
import transformers
import huggingface_hub

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.cuda.is_available())
print("Diffusers:", diffusers.__version__)
print("Transformers:", transformers.__version__)
print("Hub:", huggingface_hub.__version__)

from diffusers import AutoPipelineForText2Image

print("\nAll imports successful!")

In [ ]:
import torch
from diffusers import AutoPipelineForText2Image

MODEL_ID = "stabilityai/sd-turbo"

pipe = AutoPipelineForText2Image.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    variant="fp16"
)

pipe = pipe.to("cuda")

print("SD-Turbo loaded successfully!")

In [ ]:
import time
import torch
from IPython.display import display

prompt = """
A small red vintage car parked beside a beautiful mountain lake,
snow-covered mountains in the background, cinematic lighting,
highly detailed realistic photography
"""

generator = torch.Generator("cuda").manual_seed(42)

torch.cuda.reset_peak_memory_stats()
start = time.time()

with torch.inference_mode():
    result = pipe(
        prompt=prompt,
        height=512,
        width=512,
        num_inference_steps=4,
        guidance_scale=0.0,
        generator=generator
    )

image = result.images[0]

generation_time = time.time() - start
peak_memory = torch.cuda.max_memory_allocated() / (1024 ** 3)

image.save("/kaggle/working/generated_image.png")
display(image)

print(f"Generation time: {generation_time:.2f} seconds")
print(f"Peak GPU memory: {peak_memory:.2f} GB")

In [ ]:
import json
import os

results = {
    "model": MODEL_ID,
    "resolution": "512x512",
    "inference_steps": 4,
    "precision": "FP16",
    "generation_time_seconds": round(generation_time, 2),
    "peak_gpu_memory_gb": round(peak_memory, 3),
    "image_path": "/kaggle/working/generated_image.png"
}

with open("/kaggle/working/results.json", "w") as f:
    json.dump(results, f, indent=4)

print(json.dumps(results, indent=4))

In [ ]:
baseline = {
    "steps": 4,
    "precision": "FP16",
    "time": generation_time,
    "memory": peak_memory
}

print("Baseline saved:", baseline)

In [ ]:
pipe.enable_attention_slicing("max")

print("Attention slicing enabled")

In [ ]:
import time
import torch
from IPython.display import display

generator = torch.Generator("cuda").manual_seed(42)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

start = time.time()

with torch.inference_mode():
    optimized_result = pipe(
        prompt=prompt,
        height=512,
        width=512,
        num_inference_steps=2,
        guidance_scale=0.0,
        generator=generator
    )

optimized_image = optimized_result.images[0]

optimized_time = time.time() - start
optimized_memory = torch.cuda.max_memory_allocated() / (1024 ** 3)

optimized_image.save("/kaggle/working/optimized_image.png")
display(optimized_image)

print(f"Optimized generation time: {optimized_time:.2f} seconds")
print(f"Peak GPU memory: {optimized_memory:.3f} GB")

In [ ]:
import pandas as pd

comparison = pd.DataFrame([
    {
        "Configuration": "Baseline",
        "Steps": 4,
        "Time (seconds)": baseline["time"],
        "Peak Memory (GB)": baseline["memory"]
    },
    {
        "Configuration": "Optimized",
        "Steps": 2,
        "Time (seconds)": optimized_time,
        "Peak Memory (GB)": optimized_memory
    }
])

display(comparison)

print(
    "Latency reduction:",
    round((1 - optimized_time / baseline["time"]) * 100, 2),
    "%"
)

print(
    "Memory reduction:",
    round((1 - optimized_memory / baseline["memory"]) * 100, 2),
    "%"
)

In [ ]:
import torch
import time
import statistics
import pandas as pd
from IPython.display import display

def benchmark(steps, attention_slicing, repeats=3):

    if attention_slicing:
        pipe.enable_attention_slicing("max")
    else:
        pipe.disable_attention_slicing()

    # Warm-up
    with torch.inference_mode():
        pipe(
            prompt=prompt,
            height=512,
            width=512,
            num_inference_steps=steps,
            guidance_scale=0.0,
            generator=torch.Generator("cuda").manual_seed(42)
        )

    torch.cuda.synchronize()
    torch.cuda.empty_cache()

    times = []
    peak_memory = 0
    final_image = None

    for i in range(repeats):
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

        start = time.perf_counter()

        with torch.inference_mode():
            result = pipe(
                prompt=prompt,
                height=512,
                width=512,
                num_inference_steps=steps,
                guidance_scale=0.0,
                generator=torch.Generator("cuda").manual_seed(42)
            )

        torch.cuda.synchronize()
        times.append(time.perf_counter() - start)

        peak_memory = max(
            peak_memory,
            torch.cuda.max_memory_allocated() / (1024 ** 3)
        )

        final_image = result.images[0]
        del result

    return {
        "steps": steps,
        "attention_slicing": attention_slicing,
        "median_time": statistics.median(times),
        "peak_memory": peak_memory,
        "image": final_image
    }


results = []

results.append(benchmark(4, False))
results.append(benchmark(2, False))
results.append(benchmark(2, True))

comparison = pd.DataFrame([
    {
        "Configuration": f'{r["steps"]} steps, slicing={r["attention_slicing"]}',
        "Latency (seconds)": round(r["median_time"], 3),
        "Peak memory (GB)": round(r["peak_memory"], 3)
    }
    for r in results
])

display(comparison)

In [ ]:
baseline_time = results[0]["median_time"]
baseline_memory = results[0]["peak_memory"]

for r in results[1:]:
    latency_gain = (1 - r["median_time"] / baseline_time) * 100
    memory_gain = (1 - r["peak_memory"] / baseline_memory) * 100

    print(
        f'{r["steps"]} steps | '
        f'Slicing: {r["attention_slicing"]}'
    )
    print(f"Latency improvement: {latency_gain:.2f}%")
    print(f"Memory improvement: {memory_gain:.2f}%")
    print()

In [ ]:
import torch
import time
from PIL import Image

pipe.disable_attention_slicing()

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

generator = torch.Generator("cuda").manual_seed(42)

torch.cuda.synchronize()
start = time.perf_counter()

with torch.inference_mode():
    low_res_result = pipe(
        prompt=prompt,
        height=384,
        width=384,
        num_inference_steps=2,
        guidance_scale=0.0,
        generator=generator
    )

torch.cuda.synchronize()

low_res_time = time.perf_counter() - start
low_res_memory = torch.cuda.max_memory_allocated() / (1024 ** 3)

low_res_image = low_res_result.images[0]

# Classical CV post-processing: Lanczos upscaling
upscaled_image = low_res_image.resize(
    (512, 512),
    Image.Resampling.LANCZOS
)

upscaled_image.save("/kaggle/working/upscaled_512.png")

print("Original generated resolution:", low_res_image.size)
print("Final resolution:", upscaled_image.size)
print(f"Generation time: {low_res_time:.3f} seconds")
print(f"Peak GPU memory: {low_res_memory:.3f} GB")

display(upscaled_image)

In [ ]:
baseline_time = results[0]["median_time"]
baseline_memory = results[0]["peak_memory"]

print("BASELINE: 512x512, 4 steps")
print(f"Time: {baseline_time:.3f}s")
print(f"Memory: {baseline_memory:.3f} GB")

print("\nOPTIMIZATION 1: 512x512, 2 steps")
print(f"Time: {results[1]['median_time']:.3f}s")
print(f"Memory: {results[1]['peak_memory']:.3f} GB")

print("\nOPTIMIZATION 2: 384x384, 2 steps, upscaled to 512")
print(f"Time: {low_res_time:.3f}s")
print(f"Memory: {low_res_memory:.3f} GB")

In [ ]:
import torch
from transformers import CLIPModel, CLIPProcessor

clip_id = "openai/clip-vit-base-patch32"

clip_processor = CLIPProcessor.from_pretrained(clip_id)
clip_model = CLIPModel.from_pretrained(clip_id)

clip_model = clip_model.to("cpu")
clip_model.eval()

print("CLIP evaluator loaded!")

In [ ]:
from PIL import Image
import torch
import torch.nn.functional as F
import pandas as pd

image_paths = {
    "Baseline_4_steps": "/kaggle/working/generated_image.png",
    "Optimized_2_steps": "/kaggle/working/optimized_image.png",
    "Optimized_384_upscaled": "/kaggle/working/upscaled_512.png"
}

def calculate_clip_similarity(image_path, prompt):
    image = Image.open(image_path).convert("RGB")

    inputs = clip_processor(
        text=[prompt],
        images=image,
        return_tensors="pt",
        padding=True
    )

    with torch.inference_mode():
        outputs = clip_model(**inputs)

    image_embedding = F.normalize(outputs.image_embeds, dim=-1)
    text_embedding = F.normalize(outputs.text_embeds, dim=-1)

    similarity = (image_embedding @ text_embedding.T).item()

    return similarity

clip_results = []

for name, path in image_paths.items():
    score = calculate_clip_similarity(path, prompt)

    clip_results.append({
        "Configuration": name,
        "CLIP Similarity": round(score, 4)
    })

clip_df = pd.DataFrame(clip_results)
display(clip_df)

In [ ]:
from PIL import ImageEnhance

image = Image.open(
    "/kaggle/working/upscaled_512.png"
).convert("RGB")

image = ImageEnhance.Contrast(image).enhance(1.08)
image = ImageEnhance.Color(image).enhance(1.08)
image = ImageEnhance.Sharpness(image).enhance(1.15)

image.save("/kaggle/working/enhanced_image.png")

display(image)

print("Classical CV enhancement completed!")

In [ ]:
enhanced_score = calculate_clip_similarity(
    "/kaggle/working/enhanced_image.png",
    prompt
)

print("Enhanced image CLIP similarity:", round(enhanced_score, 4))

In [ ]:
import torch
import time
import pandas as pd
from PIL import Image

test_prompts = [
    "A red car parked to the left of a blue bicycle, beside a lake, snow-covered mountains in the background, realistic photography",

    "A wooden table with a glass vase containing yellow flowers, a closed book on the left and a cup of coffee on the right, indoor natural lighting",

    "A small white house surrounded by green trees, a river flowing in front of it, mountains in the distance, cloudy afternoon"
]

def generate_and_evaluate(prompt_text, steps, filename):
    generator = torch.Generator("cuda").manual_seed(42)

    torch.cuda.synchronize()
    start = time.perf_counter()

    with torch.inference_mode():
        output = pipe(
            prompt=prompt_text,
            height=512,
            width=512,
            num_inference_steps=steps,
            guidance_scale=0.0,
            generator=generator
        )

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    img = output.images[0]
    img.save(filename)

    score = calculate_clip_similarity(filename, prompt_text)

    return elapsed, score


evaluation_rows = []

for i, test_prompt in enumerate(test_prompts):

    baseline_time, baseline_clip = generate_and_evaluate(
        test_prompt, 4,
        f"/kaggle/working/test_{i}_baseline.png"
    )

    optimized_time, optimized_clip = generate_and_evaluate(
        test_prompt, 2,
        f"/kaggle/working/test_{i}_optimized.png"
    )

    evaluation_rows.append({
        "Prompt": f"Prompt {i+1}",
        "Baseline time": round(baseline_time, 3),
        "Optimized time": round(optimized_time, 3),
        "Baseline CLIP": round(baseline_clip, 4),
        "Optimized CLIP": round(optimized_clip, 4)
    })

evaluation_df = pd.DataFrame(evaluation_rows)
display(evaluation_df)

evaluation_df.to_csv(
    "/kaggle/working/multiprompt_evaluation.csv",
    index=False
)

In [ ]:
import pandas as pd
import json

summary = {
    "model": "stabilityai/sd-turbo",
    "gpu": "Tesla T4",
    "precision": "FP16",
    "resolution": "512x512",

    "baseline": {
        "steps": 4,
        "latency_seconds": 0.528,
        "peak_memory_gb": 3.026,
        "clip_similarity": 0.3477
    },

    "optimized": {
        "steps": 2,
        "latency_seconds": 0.372,
        "peak_memory_gb": 3.026,
        "clip_similarity": 0.3729
    },

    "low_resolution": {
        "resolution": "384x384",
        "steps": 2,
        "latency_seconds": 0.481,
        "peak_memory_gb": 2.765,
        "upscaled_resolution": "512x512",
        "clip_similarity": 0.3671
    },

    "enhancement": {
        "method": "Contrast, colour and sharpness enhancement",
        "clip_similarity": 0.3689
    },

    "multiprompt_average": {
        "baseline_latency": 0.5673,
        "optimized_latency": 0.3763,
        "baseline_clip": 0.3271,
        "optimized_clip": 0.3303
    }
}

with open("/kaggle/working/evaluation_summary.json", "w") as f:
    json.dump(summary, f, indent=4)

print("Evaluation summary saved!")

In [ ]:
import torch
import time
import gc
from diffusers import AutoPipelineForText2Image

# Preserve FP16 results and release the GPU pipeline
del pipe
gc.collect()
torch.cuda.empty_cache()

fp32_pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sd-turbo",
    torch_dtype=torch.float32
)

fp32_pipe.enable_model_cpu_offload()

torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()

start = time.perf_counter()

with torch.inference_mode():
    fp32_output = fp32_pipe(
        prompt=prompt,
        height=512,
        width=512,
        num_inference_steps=2,
        guidance_scale=0.0,
        generator=torch.Generator("cuda").manual_seed(12345)
    )

torch.cuda.synchronize()

fp32_time = time.perf_counter() - start
fp32_memory = torch.cuda.max_memory_allocated() / 1024**3

fp32_output.images[0].save(
    "/kaggle/working/sdturbo_fp32.png"
)

print("FP32 generation successful!")
print(f"Time: {fp32_time:.3f} seconds")
print(f"Peak allocated memory: {fp32_memory:.3f} GB")

In [ ]:
import os
from pathlib import Path
from huggingface_hub import snapshot_download

model_path = snapshot_download(
    repo_id="stabilityai/sd-turbo"
)

total_bytes = sum(
    f.stat().st_size
    for f in Path(model_path).rglob("*")
    if f.is_file()
)

model_size_mb = total_bytes / (1024 ** 2)
model_size_gb = total_bytes / (1024 ** 3)

print("Model directory:", model_path)
print(f"Model size: {model_size_mb:.2f} MB")
print(f"Model size: {model_size_gb:.3f} GB")

In [ ]:
import torch
import time
import statistics
import pandas as pd

pipe.disable_attention_slicing()

def final_benchmark(steps, repeats=3):
    # Warm-up
    with torch.inference_mode():
        pipe(
            prompt=prompt,
            height=512,
            width=512,
            num_inference_steps=steps,
            guidance_scale=0.0,
            generator=torch.Generator("cuda").manual_seed(12345)
        )

    torch.cuda.synchronize()
    torch.cuda.empty_cache()

    times = []
    peak_allocated = 0
    peak_reserved = 0
    final_image = None

    for i in range(repeats):
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

        start = time.perf_counter()

        with torch.inference_mode():
            output = pipe(
                prompt=prompt,
                height=512,
                width=512,
                num_inference_steps=steps,
                guidance_scale=0.0,
                generator=torch.Generator("cuda").manual_seed(12345)
            )

        torch.cuda.synchronize()
        times.append(time.perf_counter() - start)

        peak_allocated = max(
            peak_allocated,
            torch.cuda.max_memory_allocated() / 1024**3
        )

        peak_reserved = max(
            peak_reserved,
            torch.cuda.max_memory_reserved() / 1024**3
        )

        final_image = output.images[0]
        del output

    filename = f"/kaggle/working/sdturbo_{steps}_steps.png"
    final_image.save(filename)

    clip_score = calculate_clip_similarity(filename, prompt)

    return {
        "steps": steps,
        "resolution": "512x512",
        "precision": "FP16",
        "guidance_scale": 0.0,
        "seed": 12345,
        "latency_seconds": statistics.median(times),
        "mean_latency_seconds": statistics.mean(times),
        "peak_allocated_gb": peak_allocated,
        "peak_reserved_gb": peak_reserved,
        "clip_similarity": clip_score,
        "image_path": filename
    }


final_results = [
    final_benchmark(steps)
    for steps in [1, 2, 4]
]

final_df = pd.DataFrame(final_results)
display(final_df)

In [ ]:
import pandas as pd
import torch
import json
import os

MODEL_ID = "stabilityai/sd-turbo"
MODEL_SIZE_MB = 12357.07

rows = []

# Standardized FP16 experiments: seed 12345
for r in final_results:
    rows.append({
        "member": 2,
        "model_id": MODEL_ID,
        "run_id": f'FP16_{r["steps"]}_steps',
        "precision": "FP16",
        "scheduler": "Default SD-Turbo scheduler",
        "steps": r["steps"],
        "guidance_scale": r["guidance_scale"],
        "height": 512,
        "width": 512,
        "seed": 12345,
        "prompt_id": "primary_test_prompt",
        "latency_seconds": r["latency_seconds"],
        "peak_allocated_mb": r["peak_allocated_gb"] * 1024,
        "peak_reserved_mb": r["peak_reserved_gb"] * 1024,
        "model_directory_size_mb": MODEL_SIZE_MB,
        "clip_similarity": r["clip_similarity"],
        "sharpness_laplacian_var": None,
        "brightness_mean": None,
        "contrast_std": None,
        "hardware": torch.cuda.get_device_name(0),
        "notes": "FP16 GPU inference; warm-up followed by 3 timed runs"
    })

# FP32 experiment: CPU offloading
rows.append({
    "member": 2,
    "model_id": MODEL_ID,
    "run_id": "FP32_2_steps_cpu_offload",
    "precision": "FP32",
    "scheduler": "Default SD-Turbo scheduler",
    "steps": 2,
    "guidance_scale": 0.0,
    "height": 512,
    "width": 512,
    "seed": 12345,
    "prompt_id": "primary_test_prompt",
    "latency_seconds": fp32_time,
    "peak_allocated_mb": fp32_memory * 1024,
    "peak_reserved_mb": None,
    "model_directory_size_mb": MODEL_SIZE_MB,
    "clip_similarity": None,
    "sharpness_laplacian_var": None,
    "brightness_mean": None,
    "contrast_std": None,
    "hardware": torch.cuda.get_device_name(0),
    "notes": "FP32 with CPU offloading; single run; not directly comparable to FP16 timing"
})

metrics_df = pd.DataFrame(rows)

metrics_df.to_csv(
    "/kaggle/working/member2_metrics.csv",
    index=False
)

display(metrics_df)

print("Saved member2_metrics.csv")

In [ ]:
best_config = {
    "member": 2,
    "model": MODEL_ID,
    "selected_profile": "BALANCED",
    "precision": "FP16",
    "steps": 2,
    "resolution": [512, 512],
    "guidance_scale": 0.0,
    "seed": 12345,
    "measured_latency_seconds": final_results[1]["latency_seconds"],
    "peak_allocated_memory_gb": final_results[1]["peak_allocated_gb"],
    "peak_reserved_memory_gb": final_results[1]["peak_reserved_gb"],
    "clip_similarity": final_results[1]["clip_similarity"],
    "gpu": torch.cuda.get_device_name(0),
    "model_directory_size_mb": MODEL_SIZE_MB,
    "selection_reason": "Best CLIP similarity among tested FP16 step configurations with substantially lower latency than 4 steps."
}

with open("/kaggle/working/best_config.json", "w") as f:
    json.dump(best_config, f, indent=4)

print(json.dumps(best_config, indent=4))

In [ ]:
import sys
import diffusers
import transformers
import huggingface_hub
import accelerate

environment = {
    "python": sys.version,
    "pytorch": torch.__version__,
    "diffusers": diffusers.__version__,
    "transformers": transformers.__version__,
    "huggingface_hub": huggingface_hub.__version__,
    "accelerate": accelerate.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "gpu_vram_gb": round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ),
    "cuda_available": torch.cuda.is_available()
}

with open("/kaggle/working/environment.txt", "w") as f:
    for key, value in environment.items():
        f.write(f"{key}: {value}\n")

print(open("/kaggle/working/environment.txt").read())

In [ ]:
from PIL import Image, ImageDraw
from pathlib import Path

image_files = [
    ("/kaggle/working/sdturbo_4_steps.png", "Baseline: 4 steps"),
    ("/kaggle/working/sdturbo_2_steps.png", "Optimized: 2 steps"),
    ("/kaggle/working/sdturbo_1_steps.png", "Fast: 1 step"),
    ("/kaggle/working/upscaled_512.png", "384px + upscaled")
]

thumb_size = (384, 384)
margin = 20
label_height = 40

canvas = Image.new(
    "RGB",
    (2 * thumb_size[0] + 3 * margin,
     2 * (thumb_size[1] + label_height) + 3 * margin),
    "white"
)

draw = ImageDraw.Draw(canvas)

for i, (path, label) in enumerate(image_files):
    if not Path(path).exists():
        print("Missing:", path)
        continue

    img = Image.open(path).convert("RGB").resize(thumb_size)

    x = margin + (i % 2) * (thumb_size[0] + margin)
    y = margin + (i // 2) * (thumb_size[1] + label_height + margin)

    canvas.paste(img, (x, y))
    draw.text((x, y + thumb_size[1] + 8), label, fill="black")

canvas.save("/kaggle/working/member2_comparison_grid.png")
display(canvas)

print("Comparison grid saved!")

In [ ]:
import os

required_files = [
    "member2_metrics.csv",
    "best_config.json",
    "environment.txt",
    "member2_comparison_grid.png",
    "generated_image.png",
    "optimized_image.png",
    "sdturbo_1_steps.png",
    "sdturbo_2_steps.png",
    "sdturbo_4_steps.png",
    "upscaled_512.png",
    "enhanced_image.png"
]

for filename in required_files:
    path = f"/kaggle/working/{filename}"
    status = "FOUND" if os.path.exists(path) else "MISSING"
    print(f"{filename}: {status}")

In [ ]:
summary = """
LITEGEN AI - MEMBER 2 RESULTS
Model: stabilityai/sd-turbo
Hardware: NVIDIA Tesla T4
Precision: FP16
Guidance scale: 0.0

STEP EXPERIMENT (SEED 12345)
1 step:
Latency: 0.296094 seconds
CLIP: 0.351152

2 steps:
Latency: 0.382503 seconds
CLIP: 0.367103

4 steps:
Latency: 0.541530 seconds
CLIP: 0.359239

BEST BALANCED CONFIGURATION
2 steps, FP16, 512x512
CLIP: 0.367103
Latency: 0.382503 seconds
Peak allocated memory: 3.02562 GB
Peak reserved memory: 3.619141 GB

LOW RESOLUTION EXPERIMENT
384x384, 2 steps
Latency: 0.481 seconds
Peak allocated memory: 2.765 GB
Upscaled output: 512x512

FP32 EXPERIMENT
2 steps, CPU offloading
Latency: 8.279 seconds
Peak allocated GPU memory: 3.379 GB
This is not a directly comparable timing experiment.

MODEL DIRECTORY SIZE
12357.07 MB

OPTIMIZATION STRATEGIES
1. Reduced diffusion sampling steps.
2. Reduced generation resolution with Lanczos upscaling.
3. FP16 inference.

LIMITATIONS
- FP16 was used for the main experiments; FP32 used CPU offloading.
- Only three additional complex prompts were evaluated.
- No Android on-device benchmark was performed.
- No pruning or quantization was performed.
- SD-Turbo distillation is inherited from the pretrained checkpoint.
"""

with open("/kaggle/working/member2_summary.txt", "w") as f:
    f.write(summary)

print("Summary saved.")

In [ ]:
from diffusers import AutoPipelineForText2Image
import torch

pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sd-turbo",
    torch_dtype=torch.float16,
    variant="fp16"
).to("cuda")

pipe.set_progress_bar_config(disable=False)

print("SD-Turbo loaded successfully")

In [ ]:
from transformers import CLIPModel, CLIPProcessor

clip_model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32"
).to("cpu")

clip_processor = CLIPProcessor.from_pretrained(
    "openai/clip-vit-base-patch32"
)

clip_model.eval()

print("CLIP evaluator loaded successfully")

In [ ]:
import os
import gc
import json
import time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from PIL import Image, ImageFilter
from IPython.display import display

OUTPUT_DIR = "/kaggle/working"
SEED = 12345

assert torch.cuda.is_available(), "GPU is not enabled"
assert "pipe" in globals(), "SD-Turbo pipeline is missing"
assert "clip_model" in globals(), "CLIP model is missing"
assert "clip_processor" in globals(), "CLIP processor is missing"

pipe.disable_attention_slicing()
pipe.to("cuda")

print("GPU:", torch.cuda.get_device_name(0))
print("Model ready")

In [ ]:
evaluation_prompts = {
    "P01": (
        "A red vintage sports car parked beside a mountain lake, "
        "snow-covered mountains in the background, golden sunset, "
        "realistic cinematic photography"
    ),

    "P02": (
        "A small brown dog sitting on a blue sofa, a yellow pillow "
        "to its left, a wooden coffee table in front, "
        "bright modern living room, realistic photography"
    ),

    "P03": (
        "A futuristic city at night with flying vehicles, "
        "tall glass skyscrapers, neon blue and purple lights, "
        "rain-soaked streets, cyberpunk concept art"
    ),

    "P04": (
        "A white ceramic cup filled with coffee on a wooden desk, "
        "an open book behind it, a green plant on the right, "
        "warm morning sunlight coming through a window"
    ),

    "P05": (
        "An astronaut standing on the surface of Mars, "
        "a small rover to the astronaut's left, "
        "rocky red terrain, Earth visible in the dark sky, "
        "detailed science fiction illustration"
    )
}

print("Total prompts:", len(evaluation_prompts))

for prompt_id, prompt in evaluation_prompts.items():
    print(f"\n{prompt_id}: {prompt}")

In [ ]:
benchmark_records = []
generated_images = {}

def generate_image(prompt, steps, seed=12345, size=512):
    generator = torch.Generator(device="cuda").manual_seed(seed)

    return pipe(
        prompt=prompt,
        num_inference_steps=steps,
        guidance_scale=0.0,
        height=size,
        width=size,
        generator=generator
    ).images[0]


for prompt_id, prompt in evaluation_prompts.items():

    for steps in [1, 2, 4]:

        # Warm-up
        _ = generate_image(prompt, steps)

        times = []
        peak_allocated = []
        peak_reserved = []
        image = None

        for run in range(3):

            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()

            start = time.perf_counter()

            image = generate_image(prompt, steps)

            torch.cuda.synchronize()
            elapsed = time.perf_counter() - start

            allocated_gb = (
                torch.cuda.max_memory_allocated() / (1024 ** 3)
            )

            reserved_gb = (
                torch.cuda.max_memory_reserved() / (1024 ** 3)
            )

            times.append(elapsed)
            peak_allocated.append(allocated_gb)
            peak_reserved.append(reserved_gb)

        image_path = (
            f"{OUTPUT_DIR}/{prompt_id}_{steps}steps.png"
        )

        image.save(image_path)

        generated_images[(prompt_id, steps)] = image

        benchmark_records.append({
            "prompt_id": prompt_id,
            "steps": steps,
            "seed": SEED,
            "precision": "FP16",
            "resolution": "512x512",
            "guidance_scale": 0.0,
            "median_latency_s": np.median(times),
            "mean_latency_s": np.mean(times),
            "peak_allocated_gb": max(peak_allocated),
            "peak_reserved_gb": max(peak_reserved),
            "image_path": image_path
        })

        print(
            f"{prompt_id} | {steps} steps | "
            f"Median: {np.median(times):.3f}s | "
            f"Memory: {max(peak_allocated):.3f} GB"
        )


benchmark_df = pd.DataFrame(benchmark_records)

benchmark_df.to_csv(
    f"{OUTPUT_DIR}/multiprompt_benchmark.csv",
    index=False
)

display(benchmark_df)

In [ ]:
def calculate_clip_score(prompt, image):

    inputs = clip_processor(
        text=[prompt],
        images=image,
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    clip_device = next(clip_model.parameters()).device

    inputs = {
        k: v.to(clip_device)
        for k, v in inputs.items()
    }

    with torch.no_grad():
        outputs = clip_model(**inputs)

        image_features = outputs.image_embeds
        text_features = outputs.text_embeds

        image_features = torch.nn.functional.normalize(
            image_features, dim=-1
        )

        text_features = torch.nn.functional.normalize(
            text_features, dim=-1
        )

        score = (
            image_features * text_features
        ).sum(dim=-1).item()

    return score


clip_scores = []

for _, row in benchmark_df.iterrows():

    prompt_id = row["prompt_id"]
    steps = int(row["steps"])

    image = generated_images[(prompt_id, steps)]
    prompt = evaluation_prompts[prompt_id]

    score = calculate_clip_score(prompt, image)

    clip_scores.append(score)

benchmark_df["clip_score"] = clip_scores

benchmark_df.to_csv(
    f"{OUTPUT_DIR}/multiprompt_benchmark.csv",
    index=False
)

display(
    benchmark_df[
        ["prompt_id", "steps", "median_latency_s", "clip_score"]
    ]
)

In [ ]:
def image_quality_metrics(image):

    gray = np.asarray(
        image.convert("L"),
        dtype=np.float32
    )

    # Brightness
    brightness = float(gray.mean())

    # Contrast
    contrast = float(gray.std())

    # Sharpness: variance of Laplacian
    padded = np.pad(gray, 1, mode="edge")

    laplacian = (
        padded[:-2, 1:-1]
        + padded[2:, 1:-1]
        + padded[1:-1, :-2]
        + padded[1:-1, 2:]
        - 4 * gray
    )

    sharpness = float(laplacian.var())

    return {
        "brightness": brightness,
        "contrast": contrast,
        "sharpness": sharpness
    }


quality_records = []

for _, row in benchmark_df.iterrows():

    prompt_id = row["prompt_id"]
    steps = int(row["steps"])

    image = generated_images[(prompt_id, steps)]

    metrics = image_quality_metrics(image)

    quality_records.append({
        "prompt_id": prompt_id,
        "steps": steps,
        **metrics
    })


quality_df = pd.DataFrame(quality_records)

benchmark_df = benchmark_df.merge(
    quality_df,
    on=["prompt_id", "steps"]
)

benchmark_df.to_csv(
    f"{OUTPUT_DIR}/multiprompt_benchmark.csv",
    index=False
)

display(benchmark_df)

In [ ]:
fig, axes = plt.subplots(
    5, 3,
    figsize=(12, 20)
)

for row_idx, prompt_id in enumerate(evaluation_prompts):

    for col_idx, steps in enumerate([1, 2, 4]):

        image = generated_images[(prompt_id, steps)]

        axes[row_idx, col_idx].imshow(image)
        axes[row_idx, col_idx].axis("off")

        axes[row_idx, col_idx].set_title(
            f"{prompt_id} | {steps} steps"
        )

plt.tight_layout()

grid_path = f"{OUTPUT_DIR}/multiprompt_comparison_grid.png"

plt.savefig(
    grid_path,
    dpi=160,
    bbox_inches="tight"
)

plt.show()

print("Saved:", grid_path)

In [ ]:
human_eval = benchmark_df[
    ["prompt_id", "steps", "image_path"]
].copy()

human_eval["object_presence"] = np.nan
human_eval["attribute_correctness"] = np.nan
human_eval["spatial_correctness"] = np.nan
human_eval["visual_quality"] = np.nan
human_eval["failure_notes"] = ""

human_eval.to_csv(
    f"{OUTPUT_DIR}/human_evaluation.csv",
    index=False
)

display(human_eval)

print(
    "Human evaluation template saved. "
    "Fill the scores manually after inspecting the grid."
)

In [ ]:
resolution_results = []
resolution_images = {}

prompt = evaluation_prompts["P01"]

for size in [384, 512]:

    _ = generate_image(prompt, 2, size=size)

    times = []
    allocated = []
    reserved = []

    for run in range(3):

        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()

        start = time.perf_counter()

        image = generate_image(
            prompt,
            steps=2,
            seed=SEED,
            size=size
        )

        torch.cuda.synchronize()

        times.append(time.perf_counter() - start)

        allocated.append(
            torch.cuda.max_memory_allocated() / (1024 ** 3)
        )

        reserved.append(
            torch.cuda.max_memory_reserved() / (1024 ** 3)
        )

    resolution_images[size] = image

    resolution_results.append({
        "resolution": f"{size}x{size}",
        "steps": 2,
        "median_latency_s": np.median(times),
        "mean_latency_s": np.mean(times),
        "peak_allocated_gb": max(allocated),
        "peak_reserved_gb": max(reserved),
        "clip_score": calculate_clip_score(prompt, image)
    })


resolution_df = pd.DataFrame(resolution_results)

resolution_df.to_csv(
    f"{OUTPUT_DIR}/resolution_comparison.csv",
    index=False
)

display(resolution_df)

In [ ]:
img_384 = resolution_images[384]

upscaled_384 = img_384.resize(
    (512, 512),
    Image.Resampling.LANCZOS
)

native_512 = resolution_images[512]

upscaled_384.save(
    f"{OUTPUT_DIR}/384_upscaled_to_512.png"
)

native_512.save(
    f"{OUTPUT_DIR}/native_512_comparison.png"
)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

axes[0].imshow(upscaled_384)
axes[0].set_title("384 → 512 (Lanczos)")
axes[0].axis("off")

axes[1].imshow(native_512)
axes[1].set_title("Native 512")
axes[1].axis("off")

plt.tight_layout()

plt.savefig(
    f"{OUTPUT_DIR}/resolution_visual_comparison.png",
    dpi=160
)

plt.show()

print(
    "Upscaled CLIP:",
    calculate_clip_score(prompt, upscaled_384)
)

print(
    "Native CLIP:",
    calculate_clip_score(prompt, native_512)
)

In [ ]:
summary = benchmark_df.groupby("steps").agg(
    median_latency_s=("median_latency_s", "mean"),
    mean_clip_score=("clip_score", "mean"),
    mean_sharpness=("sharpness", "mean"),
    mean_brightness=("brightness", "mean"),
    mean_contrast=("contrast", "mean"),
    peak_allocated_gb=("peak_allocated_gb", "max")
).reset_index()

summary.to_csv(
    f"{OUTPUT_DIR}/final_experiment_summary.csv",
    index=False
)

display(summary)

baseline = summary.loc[
    summary["steps"] == 4
].iloc[0]

optimized = summary.loc[
    summary["steps"] == 2
].iloc[0]

speed_improvement = (
    (baseline["median_latency_s"]
     - optimized["median_latency_s"])
    / baseline["median_latency_s"]
) * 100

print(
    f"\n2-step latency reduction vs 4-step: "
    f"{speed_improvement:.2f}%"
)

print(
    "Mean CLIP (2 steps):",
    round(optimized["mean_clip_score"], 4)
)

print(
    "Mean CLIP (4 steps):",
    round(baseline["mean_clip_score"], 4)
)

In [ ]:
required_files = [
    "member2_metrics.csv",
    "best_config.json",
    "environment.txt",
    "multiprompt_benchmark.csv",
    "multiprompt_comparison_grid.png",
    "human_evaluation.csv",
    "resolution_comparison.csv",
    "resolution_visual_comparison.png",
    "final_experiment_summary.csv"
]

for filename in required_files:

    path = os.path.join(OUTPUT_DIR, filename)

    status = "FOUND" if os.path.exists(path) else "MISSING"

    print(f"{status:8} {filename}")

In [ ]:
import pandas as pd
import numpy as np

human_df = pd.read_csv(
    "/kaggle/input/datasets/srujanrbhat/human-evaluation/human_evaluation.csv"
)

score_columns = [
    "object_presence",
    "attribute_correctness",
    "spatial_correctness",
    "visual_quality"
]

# Check missing scores
print("Missing scores:")
print(human_df[score_columns].isnull().sum())

# Calculate average scores
human_df["overall_score"] = human_df[score_columns].mean(axis=1)

human_summary = human_df.groupby("steps").agg(
    object_presence=("object_presence", "mean"),
    attribute_correctness=("attribute_correctness", "mean"),
    spatial_correctness=("spatial_correctness", "mean"),
    visual_quality=("visual_quality", "mean"),
    overall_score=("overall_score", "mean")
).round(3)

display(human_summary)

human_summary.to_csv(
    "/kaggle/working/human_evaluation_summary.csv"
)

In [ ]:
comparison = benchmark_df.groupby("steps").agg(
    median_latency=("median_latency_s", "mean"),
    mean_clip=("clip_score", "mean"),
    peak_memory=("peak_allocated_gb", "max")
).reset_index()

comparison = comparison.merge(
    human_summary[["overall_score"]],
    left_on="steps",
    right_index=True
)

comparison = comparison.sort_values("steps")

display(comparison)

comparison.to_csv(
    "/kaggle/working/final_model_comparison.csv",
    index=False
)

In [ ]:
official_prompts = {
    "P01": "A red sports car parked beside a yellow bicycle under a tree, with a mountain in the background.",

    "P02": "A wooden table with a blue ceramic vase, three red apples and an open book, photographed in natural light.",

    "P03": "A small white dog sitting on a green sofa, with a brown cat sleeping beside it.",

    "P04": "A futuristic city with flying vehicles, glass skyscrapers and people walking on a wet street at night.",

    "P05": "A woman wearing a red jacket standing to the left of a blue motorcycle, with a lake behind them.",

    "P06": "A wooden cabin surrounded by pine trees, snow-covered mountains in the background and a river flowing in front.",

    "P07": "A black laptop placed on a wooden desk, with a coffee cup on its right and a notebook on its left.",

    "P08": "A realistic garden containing yellow sunflowers, purple tulips and a small stone fountain.",

    "P09": "A modern living room with a grey sofa, a round coffee table and a large window overlooking a city.",

    "P10": "A golden retriever running on a beach while two birds fly above the ocean.",

    "P11": "A red umbrella leaning against a blue chair, both placed under a large tree in a quiet park.",

    "P12": "A glass bottle filled with orange juice placed behind a white plate containing two green pears and one sliced orange."
}

prompts_df = pd.DataFrame([
    {"prompt_id": k, "prompt": v}
    for k, v in official_prompts.items()
])

print("Official prompts:", len(prompts_df))
display(prompts_df)

In [ ]:
def generate_official_image(prompt, steps, seed=12345):

    generator = torch.Generator(
        device="cuda"
    ).manual_seed(seed)

    result = pipe(
        prompt=prompt,
        num_inference_steps=steps,
        guidance_scale=0.0,
        height=512,
        width=512,
        generator=generator
    )

    return result.images[0]


def get_image_metrics(image):

    gray = np.asarray(
        image.convert("L"),
        dtype=np.float32
    )

    padded = np.pad(gray, 1, mode="edge")

    laplacian = (
        padded[:-2, 1:-1]
        + padded[2:, 1:-1]
        + padded[1:-1, :-2]
        + padded[1:-1, 2:]
        - 4 * gray
    )

    return {
        "brightness": float(gray.mean()),
        "contrast": float(gray.std()),
        "sharpness": float(laplacian.var())
    }


def get_clip_score(prompt, image):

    inputs = clip_processor(
        text=[prompt],
        images=image,
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    device = next(clip_model.parameters()).device

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    with torch.no_grad():
        outputs = clip_model(**inputs)

        img_features = torch.nn.functional.normalize(
            outputs.image_embeds, dim=-1
        )

        txt_features = torch.nn.functional.normalize(
            outputs.text_embeds, dim=-1
        )

        return (
            img_features * txt_features
        ).sum().item()

In [ ]:
official_records = []

checkpoint_path = (
    "/kaggle/working/official_12_prompt_metrics.csv"
)

image_dir = "/kaggle/working/official_12_images"
os.makedirs(image_dir, exist_ok=True)

for prompt_id, prompt in official_prompts.items():

    for steps in [1, 2, 4]:

        print(f"\nRunning {prompt_id} | {steps} steps")

        # One unrecorded warm-up
        _ = generate_official_image(prompt, steps)

        times = []
        allocated = []
        reserved = []

        final_image = None

        for run in range(3):

            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()

            start = time.perf_counter()

            final_image = generate_official_image(
                prompt, steps, seed=12345
            )

            torch.cuda.synchronize()

            elapsed = time.perf_counter() - start

            times.append(elapsed)

            allocated.append(
                torch.cuda.max_memory_allocated()
                / (1024 ** 3)
            )

            reserved.append(
                torch.cuda.max_memory_reserved()
                / (1024 ** 3)
            )

        image_path = (
            f"{image_dir}/{prompt_id}_{steps}steps.png"
        )

        final_image.save(image_path)

        clip_score = get_clip_score(prompt, final_image)
        quality = get_image_metrics(final_image)

        record = {
            "prompt_id": prompt_id,
            "prompt": prompt,
            "model_id": "stabilityai/sd-turbo",
            "run_id": f"{prompt_id}_{steps}",
            "precision": "FP16",
            "scheduler": pipe.scheduler.__class__.__name__,
            "steps": steps,
            "guidance_scale": 0.0,
            "height": 512,
            "width": 512,
            "seed": 12345,
            "median_latency_s": float(np.median(times)),
            "mean_latency_s": float(np.mean(times)),
            "peak_allocated_gb": float(max(allocated)),
            "peak_reserved_gb": float(max(reserved)),
            "clip_score": float(clip_score),
            "image_path": image_path,
            **quality
        }

        official_records.append(record)

        # Save progress after every configuration
        pd.DataFrame(official_records).to_csv(
            checkpoint_path,
            index=False
        )

        print(
            f"Median: {np.median(times):.3f}s | "
            f"CLIP: {clip_score:.4f} | "
            f"Memory: {max(allocated):.3f} GB"
        )

        gc.collect()
        torch.cuda.empty_cache()

print("\nOFFICIAL 12-PROMPT BENCHMARK COMPLETE")

In [ ]:
official_df = pd.read_csv(
    "/kaggle/working/official_12_prompt_metrics.csv"
)

summary_12 = official_df.groupby("steps").agg(
    prompts_tested=("prompt_id", "nunique"),
    median_latency=("median_latency_s", "median"),
    mean_latency=("mean_latency_s", "mean"),
    mean_clip=("clip_score", "mean"),
    peak_allocated_gb=("peak_allocated_gb", "max"),
    peak_reserved_gb=("peak_reserved_gb", "max"),
    mean_sharpness=("sharpness", "mean"),
    mean_brightness=("brightness", "mean"),
    mean_contrast=("contrast", "mean")
).reset_index()

display(summary_12)

summary_12.to_csv(
    "/kaggle/working/official_12_prompt_summary.csv",
    index=False
)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(
    12, 3,
    figsize=(12, 40)
)

for i, prompt_id in enumerate(official_prompts):

    for j, steps in enumerate([1, 2, 4]):

        image_path = (
            f"{image_dir}/{prompt_id}_{steps}steps.png"
        )

        image = Image.open(image_path)

        axes[i, j].imshow(image)
        axes[i, j].axis("off")

        axes[i, j].set_title(
            f"{prompt_id} | {steps} steps"
        )

plt.tight_layout()

grid_path = (
    "/kaggle/working/official_12_prompt_grid.png"
)

plt.savefig(
    grid_path,
    dpi=120,
    bbox_inches="tight"
)

plt.show()

print("Saved:", grid_path)

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("/kaggle/working/official_12_prompt_metrics.csv")

# Aggregate results across all 12 prompts
summary = df.groupby(
    ["steps", "precision", "height", "width"]
).agg(
    avg_latency=("median_latency_s", "mean"),
    avg_clip=("clip_score", "mean"),
    avg_sharpness=("sharpness", "mean"),
    avg_contrast=("contrast", "mean"),
    avg_brightness=("brightness", "mean"),
    peak_memory=("peak_allocated_gb", "max"),
    prompts_tested=("prompt_id", "nunique")
).reset_index()

summary = summary.sort_values("steps")

display(summary.round(4))

summary.to_csv(
    "/kaggle/working/official_12_configuration_comparison.csv",
    index=False
)

In [ ]:
# Normalize quality and speed across tested configurations
summary["quality_norm"] = (
    (summary["avg_clip"] - summary["avg_clip"].min()) /
    (summary["avg_clip"].max() - summary["avg_clip"].min() + 1e-9)
)

summary["speed_norm"] = (
    (summary["avg_latency"].max() - summary["avg_latency"]) /
    (summary["avg_latency"].max() - summary["avg_latency"].min() + 1e-9)
)

# Balanced score: equal importance to quality and speed
summary["balanced_score"] = (
    0.5 * summary["quality_norm"] +
    0.5 * summary["speed_norm"]
)

best = summary.loc[summary["balanced_score"].idxmax()]

print("BEST BALANCED CONFIGURATION")
print("---------------------------")
print(f"Steps: {int(best['steps'])}")
print(f"Precision: {best['precision']}")
print(f"Resolution: {int(best['width'])}x{int(best['height'])}")
print(f"Average latency: {best['avg_latency']:.4f} sec")
print(f"Average CLIP score: {best['avg_clip']:.4f}")
print(f"Peak allocated memory: {best['peak_memory']:.3f} GB")
print(f"Balanced score: {best['balanced_score']:.4f}")

In [ ]:
import pandas as pd
import json
import os

# Load results from all 12 official prompts
df = pd.read_csv(
    "/kaggle/working/official_12_prompt_metrics.csv"
)

# Aggregate performance across prompts
summary = df.groupby(
    ["steps", "precision", "height", "width"]
).agg(
    avg_latency=("median_latency_s", "mean"),
    avg_clip_score=("clip_score", "mean"),
    peak_memory_gb=("peak_allocated_gb", "max"),
    prompts_tested=("prompt_id", "nunique")
).reset_index()

# Normalize quality and speed
summary["quality_norm"] = (
    (summary["avg_clip_score"] - summary["avg_clip_score"].min()) /
    (summary["avg_clip_score"].max() - summary["avg_clip_score"].min() + 1e-9)
)

summary["speed_norm"] = (
    (summary["avg_latency"].max() - summary["avg_latency"]) /
    (summary["avg_latency"].max() - summary["avg_latency"].min() + 1e-9)
)

summary["balanced_score"] = (
    0.5 * summary["quality_norm"] +
    0.5 * summary["speed_norm"]
)

# Select best configuration
best = summary.loc[summary["balanced_score"].idxmax()]

best_config = {
    "model": "stabilityai/sd-turbo",
    "configuration": {
        "steps": int(best["steps"]),
        "precision": str(best["precision"]),
        "resolution": {
            "width": int(best["width"]),
            "height": int(best["height"])
        },
        "guidance_scale": 0.0
    },
    "performance": {
        "average_latency_seconds": round(float(best["avg_latency"]), 4),
        "average_clip_score": round(float(best["avg_clip_score"]), 4),
        "peak_gpu_memory_gb": round(float(best["peak_memory_gb"]), 4),
        "prompts_tested": int(best["prompts_tested"]),
        "balanced_score": round(float(best["balanced_score"]), 4)
    },
    "selection_method": {
        "metric": "balanced_score",
        "quality_weight": 0.5,
        "speed_weight": 0.5
    }
}

# Save JSON
output_path = "/kaggle/working/SDTurbo_Best_Config_12_Prompts.json"

with open(output_path, "w") as f:
    json.dump(best_config, f, indent=4)

# Display JSON in notebook output
print(json.dumps(best_config, indent=4))

print("\nSaved to:", output_path)

In [ ]:
import pandas as pd
from IPython.display import display

# Load official 12-prompt experiment
df = pd.read_csv(
    "/kaggle/working/official_12_prompt_metrics.csv"
)

# Extract CLIP similarity results
clip_results = df[[
    "prompt_id",
    "prompt",
    "steps",
    "precision",
    "height",
    "width",
    "guidance_scale",
    "seed",
    "clip_score"
]].copy()

# Save CSV
output_path = "/kaggle/working/SDTurbo_CLIP_Similarity_12_Prompts.csv"

clip_results.to_csv(output_path, index=False)

display(clip_results.round(4))
print("Saved:", output_path)

In [ ]:
import pandas as pd
import os
from IPython.display import display

# Load official experiment results
df = pd.read_csv(
    "/kaggle/working/official_12_prompt_metrics.csv"
)

# Calculate downloaded model snapshot size
model_dir = "/root/.cache/huggingface/hub/models--stabilityai--sd-turbo"

if os.path.exists(model_dir):
    model_size_bytes = sum(
        os.path.getsize(os.path.join(root, file))
        for root, _, files in os.walk(model_dir)
        for file in files
    )
else:
    raise FileNotFoundError(
        "Model snapshot directory not found. Check your Kaggle model cache."
    )

model_size_gb = model_size_bytes / (1024 ** 3)

# Aggregate peak memory by configuration
memory_results = df.groupby(
    ["steps", "precision", "height", "width", "guidance_scale"]
).agg(
    peak_allocated_memory_gb=("peak_allocated_gb", "max"),
    peak_reserved_memory_gb=("peak_reserved_gb", "max"),
    prompts_tested=("prompt_id", "nunique")
).reset_index()

memory_results.insert(0, "model", "stabilityai/sd-turbo")
memory_results.insert(1, "model_snapshot_size_gb", round(model_size_gb, 4))

# Save CSV
output_path = "/kaggle/working/SDTurbo_Model_Size_Peak_Memory.csv"

memory_results.to_csv(output_path, index=False)

print(f"Downloaded model snapshot size: {model_size_gb:.4f} GB")
display(memory_results.round(4))
print("Saved:", output_path)